# Data Preparation — Class Distribution & Sample Preview

Explores `data/merged/`, the merged output of `scripts/build_dataset.py` (see `data/merged/SOURCES.md` and `scripts/class_mapping.py` for how the three raw sources were combined and remapped).

Run this notebook from the repo root so the relative paths below resolve.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml

REPO_ROOT = Path.cwd()
DATASET_ROOT = REPO_ROOT / "data" / "merged"

with open(DATASET_ROOT / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
CLASS_NAMES = data_cfg["names"]  # whatever build_dataset.py actually included
print(f"{len(CLASS_NAMES)} classes: {CLASS_NAMES}")

## Class distribution

In [ ]:
# build_dataset.py already parses every label line while filtering classes,
# so it writes this per-annotation table directly — reading it here avoids
# re-opening ~13k individual label files on every notebook run (that direct
# parse takes single-digit seconds cold, vs <0.1s for this one CSV read).
labels_df = pd.read_csv(DATASET_ROOT / "labels_long.csv")
print(f"{len(labels_df)} annotations across {labels_df['file'].nunique()} images")
labels_df.head()

In [ ]:
instance_counts = labels_df["class_name"].value_counts().reindex(CLASS_NAMES)
images_per_class = labels_df.drop_duplicates(["class_name", "file"])["class_name"].value_counts().reindex(CLASS_NAMES)

summary = pd.DataFrame({"instances": instance_counts, "images_containing_class": images_per_class})
summary

In [ ]:
plt.figure(figsize=(10, 5))
instance_counts.plot(kind="bar", color="#4C72B0")
plt.title("Instance count per class")
plt.ylabel("Number of annotations")
plt.xlabel("Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Same counts, broken down by split — checks train/val/test aren't skewed
split_counts = labels_df.groupby(["class_name", "split"]).size().unstack(fill_value=0).reindex(CLASS_NAMES)
split_counts.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Instance count per class, by split")
plt.ylabel("Number of annotations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Same counts, broken down by source dataset — checks no single source dominates a class
source_counts = labels_df.groupby(["class_name", "source"]).size().unstack(fill_value=0).reindex(CLASS_NAMES)
source_counts.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="tab10")
plt.title("Instance count per class, by source dataset")
plt.ylabel("Number of annotations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Instances vs. images-containing-class: a big gap means that class tends
# to appear several times per image (e.g. multiple workers' helmets in one shot)
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(CLASS_NAMES))
width = 0.35
ax.bar([i - width / 2 for i in x], instance_counts.values, width, label="Instances")
ax.bar([i + width / 2 for i in x], images_per_class.values, width, label="Images containing class")
ax.set_xticks(list(x))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_ylabel("Count")
ax.set_title("Instances vs. images containing each class")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display

# The class x split matrix, in both raw counts and % of each class's own
# total — the % view is what actually reveals split imbalance (rows should
# all land close to the --split-ratios target, e.g. ~80/10/10).
split_counts = (
    labels_df.groupby(["class_name", "split"]).size()
    .unstack(fill_value=0)
    .reindex(index=CLASS_NAMES, columns=["train", "val", "test"])
)
split_counts["total"] = split_counts.sum(axis=1)
split_pct = split_counts[["train", "val", "test"]].div(split_counts["total"], axis=0) * 100

print("Instance counts by class x split:")
display(split_counts.style.background_gradient(cmap="Blues", axis=0).format("{:,}"))

print("Same, as % of each class's total (rows sum to ~100%):")
display(split_pct.style.background_gradient(cmap="Blues", axis=0).format("{:.1f}%"))

## Sample image preview per class

For a given class, grabs a random sample of images that contain it and draws all their boxes — the target class in red, everything else in grey — so you can eyeball annotation quality per class.

In [ ]:
def find_image_path(label_stem, split):
    for ext in (".jpg", ".jpeg", ".png", ".bmp"):
        candidate = DATASET_ROOT / "images" / split / f"{label_stem}{ext}"
        if candidate.exists():
            return candidate
    return None

def draw_boxes(image_path, label_path, highlight_class_id=None):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        class_id, xc, yc, bw, bh = line.split()
        class_id = int(class_id)
        xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
        x1, y1 = int(xc - bw / 2), int(yc - bh / 2)
        x2, y2 = int(xc + bw / 2), int(yc + bh / 2)
        is_target = highlight_class_id is None or class_id == highlight_class_id
        color = (255, 0, 0) if is_target else (150, 150, 150)
        thickness = 3 if is_target else 1
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(img, CLASS_NAMES[class_id], (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

In [ ]:
def preview_class(class_id, n=6, seed=0):
    class_name = CLASS_NAMES[class_id]
    candidates = labels_df[labels_df["class_id"] == class_id][["file", "split"]].drop_duplicates()
    if candidates.empty:
        print(f"No samples found for class {class_id} ({class_name})")
        return
    sample = candidates.sample(min(n, len(candidates)), random_state=seed)

    cols = 3
    rows = -(-len(sample) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        image_path = find_image_path(row["file"], row["split"])
        label_path = DATASET_ROOT / "labels" / row["split"] / f"{row['file']}.txt"
        if image_path is None:
            ax.axis("off")
            continue
        img = draw_boxes(image_path, label_path, highlight_class_id=class_id)
        ax.imshow(img)
        ax.set_title(row["file"], fontsize=8)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    fig.suptitle(f"Class {class_id}: {class_name}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Preview every class. Comment out classes you don't need — 6 images x 7
# classes renders a lot at once.
for class_id in range(len(CLASS_NAMES)):
    preview_class(class_id, n=6)